## The plug-in interfaces — CRI, CNI, CSI

Kubernetes is small at the core. Three plug-in points let it talk to whatever runtime, network, and storage you have.

**CRI — Container Runtime Interface.** A gRPC API the kubelet speaks to a local runtime, with two services: `RuntimeService` (pod/container lifecycle) and `ImageService` (pull/list images). The kubelet doesn't know if `containerd` or `CRI-O` is on the other end — it just calls CRI. **One CRI per node.**

**CNI — Container Network Interface.** A small JSON-over-stdin spec for "set up a pod's network namespace." When the kubelet creates a pod's netns, it invokes the CNI plugin, which (1) allocates a pod IP from the node's pool, (2) programs routing so the pod reaches other pods, Services, and the internet (subject to NetworkPolicy), (3) hands the IP back. Common plugins — **Calico, Cilium, Flannel, Weave, Antrea, AWS VPC CNI, Azure CNI** — differ in encapsulation (VXLAN/IP-in-IP overlay vs flat L3 vs eBPF), NetworkPolicy implementation, and performance. `kind` ships `kindnet`.

**CSI — Container Storage Interface.** Notebook 06's gRPC API to storage backends. One driver per backend (EBS, PD, NFS, Ceph, local-path).

The point of all three: Kubernetes' job is **orchestration** — placing workloads, managing lifecycle, handling failure. The actual *doing* — running containers, plumbing networks, attaching disks — is **delegated**. The same orchestration logic runs on a Raspberry Pi (k3s + containerd + flannel) or a 1000-node cloud (EKS + containerd + AWS VPC CNI + EBS CSI). On our map, the **CRI** chip sits in the Node runtime, with CNI and CSI as its siblings — the three seams where the small core meets the real world.